# Laboratorio #4 – Análisis exploratorio de datos (EDA)

**Universidad Técnica Nacional**  
**Minería de datos – ISW-911**  
**Asignación grupal (investigación)**

**Integrantes:**
- Karol Miranda Mora
- Katherine Solis Ramirez
- Dylan Jimenez Alfaro
- Josué Alvarez Villalobos

---

**Conjunto de datos:** Duración de expedientes terminados del Observatorio Judicial de Costa Rica (2026).  
**Fuente:** Portal de Datos Abiertos del Poder Judicial de Costa Rica  
https://datosabiertospj.poder-judicial.go.cr/dataset/duracion-de-expedientes-terminados-del-observatorio-judicial-de-costa-rica

**Cómo está organizado el trabajo:**

| Sección | Responsable |
|---|---|
| 1. Exploración inicial | Karol |
| 2. Limpieza de datos | Katherine |
| 3. Transformación | Dylan |
| 4. Análisis y resultados | Josué |


## 1. Exploración inicial (Karol)

### 1.1 Librerías

In [4]:
%pip install pandas numpy matplotlib seaborn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: C:\Users\karol\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/80.3 kB ? eta -:--:--
     ----- ---------------------------------- 10.2/80.3 kB ? eta -:--:--
     -------------- ----------------------- 30.7/80.3 kB 435.7 kB/s eta 0:00:01
     ----------------------------- -------- 61.4/80.3 kB 465.5 kB/s eta 0:00:01
     -------------------------------------- 80.3/80.3 kB 496.6 kB/s eta 0:00:00
     ---------------------------------------- 0.0/142.6 kB ? eta -:--:--
     -- ------------------------------------- 10.2/142.6 kB ? eta -:--:--
     -- ------------------------------------- 10.2/142.6 kB ? eta -:--:--
     -------------------------------------- 142.6/142.6 kB 1.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   ---------------------------------------- 0.1/11.0 MB 1.7 MB/s eta 0:00:07
   ---------------------------------------- 0.1/11.0 MB 1.1 MB/s eta 0:00:11
    --------------------------------------- 0.2/11.0 MB 1.1 MB/s eta 0:00:10
    ---------

### 1.2 Lectura del conjunto de datos desde Git

El archivo se subió al repositorio de GitHub (público) en formato `.csv.gz`, que es un CSV comprimido. El archivo original en Excel pesa 31 MB y el CSV sin comprimir unos 91 MB, así que se prefirió comprimirlo (queda en unos 6 MB) para que el notebook cargue rápido.

Pandas lee el `.csv.gz` directamente desde la URL. Las columnas de códigos se leen como texto para no perder los ceros a la izquierda (por ejemplo `0215`).

In [5]:
URL = "https://raw.githubusercontent.com/Kyaroruki/Laboratorio-4-Mineria-Datos/main/datos/duracion_terminados.csv.gz"
RUTA_LOCAL = "datos/duracion_terminados.csv.gz"   # respaldo si no hay internet

# Los códigos se leen como texto para conservar los ceros a la izquierda
tipos = {"CodClaseAsu": "str", "CodProcedimiento": "str", "CodFase": "str"}

try:
    df = pd.read_csv(URL, compression="gzip", dtype=tipos)
    print("Datos leídos desde GitHub")
except Exception as e:
    print(f"No se pudo leer desde GitHub ({type(e).__name__}). Se usa el archivo local.")
    df = pd.read_csv(RUTA_LOCAL, compression="gzip", dtype=tipos)

print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")

No se pudo leer desde GitHub (HTTPError). Se usa el archivo local.
Filas: 309,635 | Columnas: 23


### 1.3 Funciones básicas para ver las características del conjunto de datos

In [6]:
# Primeras filas
df.head()

,CodCircuito,DescCircuito,DescMateria,DescTipoDespacho,CodDespacho,DescDespacho,Año,Numero_Mes,CodClaseAsu,DescClaseAsu,CodProcedimiento,Procedimiento,DescEstadoAsunto,AgrEstadoAsunto,CodFase,Fase,TermiMedAlterna,TermiConciliacion,Cant,DurDias,Completitud,Desc_Delito,Ref_Civil
0,11,II CIRCUITO JUDICIAL DE LIMÓN (POCOCÍ),PENAL,FISCALÍA,485,FISCALÍA ADJUNTA DEL II CIRCUITO JUDICIAL DE L...,2026,3,0215,DENUNCIA EN DESPACHO,NaN,ORDINARIO,ACUS. FISCAL (SIN ACIV. RESAR.),Acusación,NaN,Etapa Preparatoria,No,NaN,1,615.00,COMPLETA,INCUMPLIMIENTO O ABUSO PATRIA POTESTAD,NaN
1,11,II CIRCUITO JUDICIAL DE LIMÓN (POCOCÍ),PENAL,FISCALÍA,485,FISCALÍA ADJUNTA DEL II CIRCUITO JUDICIAL DE L...,2026,3,0215,DENUNCIA EN DESPACHO,NaN,ORDINARIO,INCOMPETENCIA (JURISDICCIONAL),NaN,NaN,Etapa Preparatoria,No,NaN,1,0.00,COMPLETA,INTRODUCCIÓN DE DROGA DEL CENTRO PENITENCIARIO,NaN
2,11,II CIRCUITO JUDICIAL DE LIMÓN (POCOCÍ),PENAL,FISCALÍA,485,FISCALÍA ADJUNTA DEL II CIRCUITO JUDICIAL DE L...,2026,3,0215,DENUNCIA EN DESPACHO,NaN,ORDINARIO,INCOMPETENCIA (JURISDICCIONAL),NaN,NaN,Etapa Preparatoria,No,NaN,1,"1,409.00",COMPLETA,PREVARICATO,NaN
3,11,II CIRCUITO JUDICIAL DE LIMÓN (POCOCÍ),PENAL,FISCALÍA,485,FISCALÍA ADJUNTA DEL II CIRCUITO JUDICIAL DE L...,2026,3,0215,DENUNCIA EN DESPACHO,NaN,ORDINARIO,REMITIDO OTRO DESPACHO DEL MINISTERIO PÚBLICO ...,NaN,NaN,Etapa Preparatoria,No,NaN,1,10.00,COMPLETA,AGRESIÓN CON ARMAS,NaN
4,11,II CIRCUITO JUDICIAL DE LIMÓN (POCOCÍ),PENAL,FISCALÍA,485,FISCALÍA ADJUNTA DEL II CIRCUITO JUDICIAL DE L...,2026,3,0215,DENUNCIA EN DESPACHO,NaN,ORDINARIO,"SOL. DESESTIMACIÓN, ART. 299 (ELEMENTOS PROBAT...",NaN,NaN,Etapa Intermedia,No,NaN,1,16.00,COMPLETA,AMENAZA A UN FUNCIONARIO PÚBLICO,NaN


In [7]:
# Últimas filas
df.tail()

,CodCircuito,DescCircuito,DescMateria,DescTipoDespacho,CodDespacho,DescDespacho,Año,Numero_Mes,CodClaseAsu,DescClaseAsu,CodProcedimiento,Procedimiento,DescEstadoAsunto,AgrEstadoAsunto,CodFase,Fase,TermiMedAlterna,TermiConciliacion,Cant,DurDias,Completitud,Desc_Delito,Ref_Civil
309630,9,I Circuito Judicial de Alajuela,VIOLENCIA DOMÉSTICA,Juzgado,1473,JUZGADO CONTRAVENCIONAL DE OROTINA (VIOLENCIA ...,2026,9,1111,VIOLENCIA DOMÉSTICA,NaN,VIOLENCIA DOMÉSTICA ORDINARIO ESCRITO,RECHAZO DE PLANO,NaN,NaN,NaN,NaN,No,2,71.00,COMPLETA,V.D. AMENAZAS AGRAVADAS,NaN
309631,9,I Circuito Judicial de Alajuela,VIOLENCIA DOMÉSTICA,Juzgado,1477,JUZGADO CONTRAVENCIONAL DE POÁS (VIOLENCIA DOM...,2026,5,1111,VIOLENCIA DOMÉSTICA,NaN,VIOLENCIA DOMÉSTICA (ORDINARIO),LEVANTA MEDIDA (POR VENCIMIENTO DEL PLAZO),NaN,NaN,NaN,NaN,No,107,"47,679.00",INCOMPLETA,IGNORADO,NaN
309632,9,I Circuito Judicial de Alajuela,VIOLENCIA DOMÉSTICA,Juzgado,1477,JUZGADO CONTRAVENCIONAL DE POÁS (VIOLENCIA DOM...,2026,5,3186,PROCESO DE PROTECCIÓN PARA PERSONA ADULTA MAYOR,NaN,SIN PARENTESCO,LEVANTA MEDIDA (POR VENCIMIENTO DEL PLAZO),NaN,NaN,NaN,NaN,No,2,876.00,INCOMPLETA,IGNORADO,NaN
309633,9,I Circuito Judicial de Alajuela,VIOLENCIA DOMÉSTICA,Juzgado,1477,JUZGADO CONTRAVENCIONAL DE POÁS (VIOLENCIA DOM...,2026,7,1091,TESTIMONIO DE PIEZAS,NaN,VD PARA MP POR INCUMPLIMIENTO MEDIDA DESOBEDIE...,TESTIMONIOS DE PIEZAS REMITIDOS A OTRAS OFICINAS,NaN,NaN,NaN,NaN,No,1,11.00,COMPLETA,DESCONOCIDO,NaN
309634,9,I Circuito Judicial de Alajuela,VIOLENCIA DOMÉSTICA,Juzgado,1477,JUZGADO CONTRAVENCIONAL DE POÁS (VIOLENCIA DOM...,2026,9,1111,VIOLENCIA DOMÉSTICA,NaN,VIOLENCIA DOMÉSTICA (ORDINARIO) MUJERES VÍCTIM...,LEVANTA MEDIDA (POR VENCIMIENTO DEL PLAZO),NaN,NaN,NaN,NaN,No,1,428.00,COMPLETA,IGNORADO,NaN


In [8]:
# Cantidad de filas y columnas
print("Forma del dataset (filas, columnas):", df.shape)

Forma del dataset (filas, columnas): (309635, 23)


In [9]:
# Tipos de dato, cantidad de valores no nulos y memoria usada
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 309635 entries, 0 to 309634
Data columns (total 23 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   CodCircuito        309635 non-null  int64  
 1   DescCircuito       309635 non-null  str    
 2   DescMateria        309635 non-null  str    
 3   DescTipoDespacho   309635 non-null  str    
 4   CodDespacho        309635 non-null  int64  
 5   DescDespacho       309635 non-null  str    
 6   Año                309635 non-null  int64  
 7   Numero_Mes         309635 non-null  int64  
 8   CodClaseAsu        309581 non-null  str    
 9   DescClaseAsu       309038 non-null  str    
 10  CodProcedimiento   0 non-null       str    
 11  Procedimiento      309635 non-null  str    
 12  DescEstadoAsunto   309635 non-null  str    
 13  AgrEstadoAsunto    85216 non-null   str    
 14  CodFase            0 non-null       str    
 15  Fase               276887 non-null  str    
 16  TermiMedAlter

In [10]:
# Estadísticas de las columnas numéricas
df.describe()

,CodCircuito,CodDespacho,Año,Numero_Mes,Cant,DurDias
count,"309,635.00","309,635.00","309,635.00","309,635.00","309,635.00","309,362.00"
mean,16.28,882.22,"2,026.00",5.06,2.86,"1,357.84"
std,17.94,546.19,0.00,2.56,15.45,"21,223.85"
min,1.00,4.00,"2,026.00",1.00,1.00,0.00
25%,4.00,472.00,"2,026.00",3.00,1.00,22.00
50%,10.00,687.00,"2,026.00",5.00,1.00,98.00
75%,18.00,"1,260.00","2,026.00",7.00,2.00,414.00
max,72.00,"2,106.00","2,026.00",9.00,"1,871.00","3,155,804.00"


In [11]:
# Resumen de las columnas de texto
df.describe(include=["object", "string"]).T

,count,unique,top,freq
DescCircuito,309635,79,I CIRCUITO JUDICIAL DE SAN JOSÉ,35065
DescMateria,309635,23,PENAL,170009
DescTipoDespacho,309635,71,FISCALÍA,84919
DescDespacho,309635,907,FISCALÍA ADJUNTA DEL I CIRCUITO JUDICIAL DE AL...,3395
CodClaseAsu,309581,726,0215,32538
DescClaseAsu,309038,617,DENUNCIA EN DESPACHO,36142
CodProcedimiento,0,0,NaN,NaN
Procedimiento,309635,569,ORDINARIO,154319
DescEstadoAsunto,309635,642,"SOL. DESESTIMACIÓN, ART. 299 (ELEMENTOS PROBAT...",18391
AgrEstadoAsunto,85216,45,Acusación,17430


In [12]:
# Cantidad de valores distintos por columna
df.nunique().sort_values(ascending=False)

DurDias              11515
CodDespacho            926
DescDespacho           907
CodClaseAsu            726
DescEstadoAsunto       642
Desc_Delito            635
DescClaseAsu           617
Procedimiento          569
Cant                   344
Fase                   131
DescCircuito            79
DescTipoDespacho        71
CodCircuito             65
AgrEstadoAsunto         45
DescMateria             23
Numero_Mes               9
Completitud              2
TermiConciliacion        2
TermiMedAlterna          2
Año                      1
Ref_Civil                1
CodFase                  0
CodProcedimiento         0
dtype: int64

In [13]:
# Una muestra al azar
df.sample(5, random_state=42)

,CodCircuito,DescCircuito,DescMateria,DescTipoDespacho,CodDespacho,DescDespacho,Año,Numero_Mes,CodClaseAsu,DescClaseAsu,CodProcedimiento,Procedimiento,DescEstadoAsunto,AgrEstadoAsunto,CodFase,Fase,TermiMedAlterna,TermiConciliacion,Cant,DurDias,Completitud,Desc_Delito,Ref_Civil
184325,5,CIRCUITO JUDICIAL DE HEREDIA,LABORAL,TRIBUNAL DE TRABAJO,809,TRIBUNAL DE APELACIÓN CIVIL Y TRABAJO DE HERED...,2026,5,JT001,ORDINARIO DE DESPIDO-PRIVADO,NaN,DE CESANTÍA U OTROS EXTREMOS,SENTENCIA,NaN,NaN,4. EN EJECUCION,NaN,No,1,115.00,COMPLETA,NO DELITOS,NaN
178220,47,I CIRCUITO JUDICIAL DE LIMÓN (BATÁN-MATINA),PENAL,FISCALÍA,1521,FISCALÍA DE BATÁN (PENAL),2026,3,0216,DENUNCIA POR ESCRITO,NaN,ORDINARIO,"SOL. DESESTIMACIÓN, ART. 299 (ELEMENTOS PROBAT...",NaN,NaN,Etapa Intermedia,No,NaN,1,169.00,COMPLETA,DIFUSIÓN DE PORNOGRAFÍA,NaN
294564,5,CIRCUITO JUDICIAL DE HEREDIA,PENAL JUVENIL,FISCALÍA PENAL JUVENIL,811,FISCALÍA ADJUNTA DE HEREDIA (PENAL JUVENIL),2026,2,0215,DENUNCIA EN DESPACHO,NaN,ESPECIAL,SOB.DEF. PRESCRIP.ART.31 C.P.P.,Sobreseimiento definitivo,NaN,Fase Conclusiva de la Investigación,No,NaN,1,242.00,COMPLETA,DESÓRDENES,NaN
162213,42,CIRCUITO JUDICIAL DE PUNTARENAS (GARABITO),TRÁNSITO,JUZGADO DE TRÁNSITO,1598,JUZGADO CONTRAVENCIONAL DE GARABITO (TRÁNSITO),2026,4,0540,COLISIÓN ENTRE VEHÍCULOS-ACCIDENTE SIMPLE,NaN,COLISIÓN,INCOMPETENCIA,Incompentencia,NaN,1. Inicial,NaN,No,7,211.00,COMPLETA,DESCONOCIDO,NaN
79087,11,Circuito Judicial de Heredia,VIOLENCIA DOMÉSTICA,Juzgado,1612,"JUZGADO DE PENSIONES, CONTRA LA VIOLENCIA DOMÉ...",2026,8,1111,VIOLENCIA DOMÉSTICA,NaN,VIOLENCIA DOMÉSTICA ORDINARIO VERBAL (MUJERES ...,INCOMPETENCIA,Incompentencia,NaN,NaN,NaN,No,1,1.00,COMPLETA,DESCONOCIDO,NaN


**Primeras observaciones**
- El dataset tiene 309 635 filas y 23 columnas, cada fila es un grupo de expedientes terminados con su duración en días.
- Hay columnas numéricas (`DurDias`, `Cant`, `Numero_Mes`, `Año`, `CodCircuito`, `CodDespacho`) y muchas de texto.
- `Año` tiene un solo valor (2026), así que no aporta información.
- `CodProcedimiento` y `CodFase` vienen totalmente vacías.
- `DurDias` tiene valores muy altos frente a su mediana, esto se analiza en la sección de valores atípicos.

### 1.4 Cambio de nombres de las columnas a español

Los nombres originales ya estaban en español pero abreviados (`Cant`, `DurDias`, `TermiMedAlterna`) y mezclaban mayúsculas, guiones bajos y una `ñ` (`Año`). Se pasaron a nombres completos en minúscula y con guion bajo, para que sean fáciles de leer y de escribir en el código. La `ñ` de `Año` se cambió a `anio` para evitar problemas de codificación.

In [14]:
nombres_es = {
    "CodCircuito": "codigo_circuito",
    "DescCircuito": "circuito",
    "DescMateria": "materia",
    "DescTipoDespacho": "tipo_despacho",
    "CodDespacho": "codigo_despacho",
    "DescDespacho": "despacho",
    "Año": "anio",
    "Numero_Mes": "mes",
    "CodClaseAsu": "codigo_clase_asunto",
    "DescClaseAsu": "clase_asunto",
    "CodProcedimiento": "codigo_procedimiento",
    "Procedimiento": "procedimiento",
    "DescEstadoAsunto": "estado_asunto",
    "AgrEstadoAsunto": "estado_asunto_agrupado",
    "CodFase": "codigo_fase",
    "Fase": "fase",
    "TermiMedAlterna": "termina_medida_alterna",
    "TermiConciliacion": "termina_conciliacion",
    "Cant": "cantidad",
    "DurDias": "duracion_dias",
    "Completitud": "completitud",
    "Desc_Delito": "delito",
    "Ref_Civil": "referencia_civil",
}

df = df.rename(columns=nombres_es)
print(df.columns.tolist())

['codigo_circuito', 'circuito', 'materia', 'tipo_despacho', 'codigo_despacho', 'despacho', 'anio', 'mes', 'codigo_clase_asunto', 'clase_asunto', 'codigo_procedimiento', 'procedimiento', 'estado_asunto', 'estado_asunto_agrupado', 'codigo_fase', 'fase', 'termina_medida_alterna', 'termina_conciliacion', 'cantidad', 'duracion_dias', 'completitud', 'delito', 'referencia_civil']


In [15]:
df.head()

,codigo_circuito,circuito,materia,tipo_despacho,codigo_despacho,despacho,anio,mes,codigo_clase_asunto,clase_asunto,codigo_procedimiento,procedimiento,estado_asunto,estado_asunto_agrupado,codigo_fase,fase,termina_medida_alterna,termina_conciliacion,cantidad,duracion_dias,completitud,delito,referencia_civil
0,11,II CIRCUITO JUDICIAL DE LIMÓN (POCOCÍ),PENAL,FISCALÍA,485,FISCALÍA ADJUNTA DEL II CIRCUITO JUDICIAL DE L...,2026,3,0215,DENUNCIA EN DESPACHO,NaN,ORDINARIO,ACUS. FISCAL (SIN ACIV. RESAR.),Acusación,NaN,Etapa Preparatoria,No,NaN,1,615.00,COMPLETA,INCUMPLIMIENTO O ABUSO PATRIA POTESTAD,NaN
1,11,II CIRCUITO JUDICIAL DE LIMÓN (POCOCÍ),PENAL,FISCALÍA,485,FISCALÍA ADJUNTA DEL II CIRCUITO JUDICIAL DE L...,2026,3,0215,DENUNCIA EN DESPACHO,NaN,ORDINARIO,INCOMPETENCIA (JURISDICCIONAL),NaN,NaN,Etapa Preparatoria,No,NaN,1,0.00,COMPLETA,INTRODUCCIÓN DE DROGA DEL CENTRO PENITENCIARIO,NaN
2,11,II CIRCUITO JUDICIAL DE LIMÓN (POCOCÍ),PENAL,FISCALÍA,485,FISCALÍA ADJUNTA DEL II CIRCUITO JUDICIAL DE L...,2026,3,0215,DENUNCIA EN DESPACHO,NaN,ORDINARIO,INCOMPETENCIA (JURISDICCIONAL),NaN,NaN,Etapa Preparatoria,No,NaN,1,"1,409.00",COMPLETA,PREVARICATO,NaN
3,11,II CIRCUITO JUDICIAL DE LIMÓN (POCOCÍ),PENAL,FISCALÍA,485,FISCALÍA ADJUNTA DEL II CIRCUITO JUDICIAL DE L...,2026,3,0215,DENUNCIA EN DESPACHO,NaN,ORDINARIO,REMITIDO OTRO DESPACHO DEL MINISTERIO PÚBLICO ...,NaN,NaN,Etapa Preparatoria,No,NaN,1,10.00,COMPLETA,AGRESIÓN CON ARMAS,NaN
4,11,II CIRCUITO JUDICIAL DE LIMÓN (POCOCÍ),PENAL,FISCALÍA,485,FISCALÍA ADJUNTA DEL II CIRCUITO JUDICIAL DE L...,2026,3,0215,DENUNCIA EN DESPACHO,NaN,ORDINARIO,"SOL. DESESTIMACIÓN, ART. 299 (ELEMENTOS PROBAT...",NaN,NaN,Etapa Intermedia,No,NaN,1,16.00,COMPLETA,AMENAZA A UN FUNCIONARIO PÚBLICO,NaN


---
## 2. Limpieza de datos (Katherine)

Trabajar con el DataFrame `df` ya con los nombres en español (por ejemplo `duracion_dias`, `cantidad`, `fase`).

Pendiente:
- [ ] Identificar valores nulos
- [ ] Gráficos de valores nulos
- [ ] Identificar valores atípicos
- [ ] Gráficos de valores atípicos
- [ ] Funciones para imputar o tratar nulos y atípicos, con la justificación de cada decisión en la documentación

*Nota:* además de los nulos de Python (`NaN`), revisar si hay valores que funcionan como nulos pero están escritos como texto.

---
## 3. Transformación (Dylan)

Pendiente:
- [ ] Corregir tipos de datos
- [ ] Convertir variables categóricas a numéricas
- [ ] Normalización o estandarización

---
## 4. Análisis y resultados (Josué)

Pendiente:
- [ ] Correlaciones y gráficos
- [ ] Variable dependiente
- [ ] Modelo propuesto (clasificación o regresión) y justificación
- [ ] Guardar el dataset final como `data_process.csv`